Tratamiento del Dataset de Caracteristicas



In [1]:
# @title 🧪 SECCIÓN 1 — Introducción

from IPython.display import display, HTML

display(HTML("""
<h2 style='color:#2E86C1;'>Aduana de Calidad de Datos - Dataset Características</h2>
<p>Este notebook implementa un proceso de validación y limpieza de datos orientado a QA.</p>
<ul>
<li>Control de nulos</li>
<li>Normalización de datos</li>
<li>Profiling automático</li>
<li>Detección de outliers</li>
</ul>
"""))

In [2]:
# @title 📥 SECCIÓN 2 — Carga manual del dataset

import pandas as pd
import numpy as np

from google.colab import files

print("📂 Subí el archivo caracteristicas_limpio.xlsx")

uploaded = files.upload()

# Obtener nombre del archivo subido
file_name = list(uploaded.keys())[0]

# Leer el archivo
df = pd.read_excel(file_name)

print("✅ Dataset cargado correctamente")
df.head()

📂 Subí el archivo caracteristicas_limpio.xlsx


Saving Caracteristicas_Limpio.xlsx to Caracteristicas_Limpio.xlsx
✅ Dataset cargado correctamente


,anio,provincia,departamento,sector,ambito,localizacion,electricidadsi,electricidadredpublica,electricidadgrupoelectrogeno,electricidadpanelfotovoltaicosolar,...,sistemadegestionescolarplanilladecalculo,disponedesalaolaboratoriodeinformaticasi,laboratoriofuncionaesespacioexclusivosi,bibliotecadisponedealmenosunasi,bibliotecafuncionaesespacioexclusivosi,subvencionestataljardinmaternalsi,subvencionestataljardindeinfantessi,subvencionestatalprimariasi,subvencionestatalsecundariasi,subvencionestatalsnusi
0,2011,Buenos Aires,25 DE MAYO,Estatal,Rural,50,46,41,,,...,,9,,,,,,,,
1,2011,Buenos Aires,25 DE MAYO,Estatal,Urbano,41,29,29,,,...,,11,,,,,,,,
2,2011,Buenos Aires,25 DE MAYO,Privado,Urbano,12,5,5,,,...,,5,,,,,2,2,3,
3,2011,Buenos Aires,9 DE JULIO,Estatal,Rural,57,46,44,,,...,,12,,,,,,,,
4,2011,Buenos Aires,9 DE JULIO,Estatal,Urbano,42,32,31,1,,...,,11,,,,,,,,


In [4]:
# @title 🔍 SECCIÓN 3 — Análisis de nulos (versión QA corregida)

from IPython.display import display, HTML

# Reemplazar vacíos y espacios por NaN
df = df.replace(r'^\s*$', np.nan, regex=True)

# Calcular nulos
nulls = df.isnull().sum().sort_values(ascending=False)
nulls = nulls[nulls > 0]

# HTML con scroll
html = f"""
<div style='height:300px; overflow-y:scroll; border:1px solid #ccc; padding:10px;'>
<h3 style='color:#C0392B;'>Columnas con valores nulos</h3>
<ul>
{''.join([f"<li><b>{col}</b>: {val}</li>" for col, val in nulls.items()])}
</ul>
</div>
"""

display(HTML(html))

print(f"Total columnas con nulos: {len(nulls)}")
print(f"Total valores nulos en dataset: {df.isnull().sum().sum()}")

Total columnas con nulos: 61
Total valores nulos en dataset: 388116


In [5]:
# @title 🧹 SECCIÓN 4 — Tratamiento de nulos (versión QA mejorada)

# Contar nulos antes
nulos_antes = df.isnull().sum().sum()

print(f"🔎 Nulos antes del tratamiento: {nulos_antes}")

# Reemplazo de nulos por 0
df = df.fillna(0)

# Contar nulos después
nulos_despues = df.isnull().sum().sum()

print(f"🧹 Nulos después del tratamiento: {nulos_despues}")

# Validación QA
assert nulos_despues == 0, "⚠️ Aún hay valores nulos en el dataset"

print("✅ Validación OK: No hay nulos en el dataset")

# Extra QA: mostrar cuántos valores se limpiaron
print(f"📊 Total de valores imputados: {nulos_antes}")

🔎 Nulos antes del tratamiento: 388116
🧹 Nulos después del tratamiento: 0
✅ Validación OK: No hay nulos en el dataset
📊 Total de valores imputados: 388116


In [6]:
# @title 🔤 SECCIÓN 5 — Normalización de texto

cols_texto = df.select_dtypes(include=['object']).columns

def formatear_texto(texto):
    if isinstance(texto, str):
        return texto.title()
    return texto

for col in cols_texto:
    df[col] = df[col].apply(formatear_texto)

print("Columnas de texto normalizadas:")
print(list(cols_texto))

Columnas de texto normalizadas:
['provincia', 'departamento', 'sector', 'ambito', 'electricidadsi', 'electricidadredpublica', 'electricidadgrupoelectrogeno', 'electricidadpanelfotovoltaicosolar', 'electricidadgeneradoreolico', 'electricidadgeneradorhidraulico', 'electricidadotro', 'equipamientoestablecimientotelevisor', 'equipamientoestablecimientosistemamultimediaocanon', 'equipamientoestablecimientoscanner', 'equipamientoestablecimientowebcam', 'equipamientoestablecimientoreproductordecd', 'equipamientoestablecimientoreproductordedvd', 'equipamientoestablecimientoimpresora', 'equipamientoestablecimientoequipoemisorderadioamfm', 'equipamientoestablecimientoequiporeceptorderadioamfm', 'equipamientoestablecimientoservidorparausoescolar', 'equipamientoestablecimientoimpresora3d', 'equipamientoestablecimientoequipodesonido', 'equipamientoestablecimientopizarrasdigitales', 'equipamientobibliotecatelevisor', 'equipamientobibliotecasistemamultimediaocanon', 'equipamientobibliotecascanner', '

In [8]:
# @title 🚨 SECCIÓN 6 — Detección de Outliers

from scipy import stats

numeric_df = df.select_dtypes(include=[np.number])

z_scores = np.abs(stats.zscore(numeric_df))

outliers = (z_scores > 3)

outliers_count = np.sum(outliers)

print("Cantidad total de valores outliers:", outliers_count)

# Mostrar columnas con más outliers
outliers_por_col = pd.Series(outliers.sum(axis=0), index=numeric_df.columns)
outliers_por_col = outliers_por_col.sort_values(ascending=False)

display(outliers_por_col.head(10))

Cantidad total de valores outliers: 332


,0
localizacion,332
anio,0


In [9]:
# @title 💾 SECCIÓN 7 — Exportación y descarga directa

from google.colab import files

file_name = "caracteristicas_final_QA.xlsx"

# Guardar archivo temporalmente
df.to_excel(file_name, index=False)

# Descargar automáticamente
files.download(file_name)

print("✅ Dataset exportado y descargado correctamente")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Dataset exportado y descargado correctamente


Tratamiento del Dataset de Matriculas

In [10]:
# @title 🧪 SECCIÓN 1 — Introducción

from IPython.display import display, HTML

display(HTML("""
<h2 style='color:#2E86C1;'>Aduana de Calidad de Datos - Dataset Matrícula</h2>
<p>Este proceso valida la integridad y consistencia del dataset de matrícula.</p>
<ul>
<li>Control de nulos</li>
<li>Normalización de texto</li>
<li>Formateo de métricas</li>
<li>Detección de outliers</li>
</ul>
"""))

In [28]:
# @title 📥 SECCIÓN 2 — Carga del dataset

import pandas as pd
import numpy as np

from google.colab import files

print("📂 Subí el archivo matricula_limpia.xlsx")

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_excel(file_name)

print("✅ Dataset cargado correctamente")
df.head()

📂 Subí el archivo matricula_limpia.xlsx


Saving Matricula_Limpio.xlsx to Matricula_Limpio (1).xlsx
✅ Dataset cargado correctamente


,anio,provincia,departamento,sector,ambito,total_grado_1,total_grado_2,total_grado_3,total_grado_4,total_grado_5,...,tasa_sobreedad_anio_3,mujeres_anio_4,tasa_repitencia_anio_4,tasa_sobreedad_anio_4,mujeres_anio_5,tasa_repitencia_anio_5,tasa_sobreedad_anio_5,mujeres_anio_6,tasa_repitencia_anio_6,tasa_sobreedad_anio_6
0,2011,Buenos Aires,25 DE MAYO,Estatal,Rural,120,119,158,118,125,...,44.067797,35,0.000000,34.444444,29,0.000000,31.764706,30,0.000000,37.894737
1,2011,Buenos Aires,25 DE MAYO,Estatal,Urbano,328,337,336,323,371,...,36.963696,148,21.223022,46.402878,141,4.824561,38.157895,103,0.000000,22.807018
2,2011,Buenos Aires,25 DE MAYO,Privado,Urbano,167,150,172,156,141,...,17.475728,60,3.000000,28.000000,45,1.219512,18.292683,49,0.000000,14.285714
3,2011,Buenos Aires,9 DE JULIO,Estatal,Rural,165,169,126,122,143,...,21.000000,17,23.913043,36.956522,30,0.000000,48.780488,16,3.703704,33.333333
4,2011,Buenos Aires,9 DE JULIO,Estatal,Urbano,551,550,528,536,566,...,35.496957,257,10.154525,41.280353,179,4.216867,40.662651,165,4.248366,34.313725


In [29]:
# @title 🔍 SECCIÓN 3 — Análisis de nulos

from IPython.display import display, HTML

# Convertir vacíos a NaN
df = df.replace(r'^\s*$', np.nan, regex=True)

nulls = df.isnull().sum().sort_values(ascending=False)
nulls = nulls[nulls > 0]

html = f"""
<div style='height:300px; overflow-y:scroll; border:1px solid #ccc; padding:10px;'>
<h3 style='color:#C0392B;'>Columnas con valores nulos</h3>
<ul>
{''.join([f"<li><b>{col}</b>: {val}</li>" for col, val in nulls.items()])}
</ul>
</div>
"""

display(HTML(html))

print(f"Total columnas con nulos: {len(nulls)}")
print(f"Total valores nulos: {df.isnull().sum().sum()}")

Total columnas con nulos: 1
Total valores nulos: 1


In [30]:
# @title 🧹 SECCIÓN 4 — Tratamiento de nulos (Imputación inteligente)

# Contar nulos antes
nulos_antes = df.isnull().sum().sum()
print(f"🔎 Nulos antes: {nulos_antes}")

# Separar columnas
cols_texto = df.select_dtypes(include=['object']).columns
cols_numericas = df.select_dtypes(include=[np.number]).columns

# 👉 1. Texto → reemplazar por valor más frecuente (moda)
for col in cols_texto:
    if df[col].isnull().sum() > 0:
        moda = df[col].mode()
        if not moda.empty:
            df[col].fillna(moda[0], inplace=True)

# 👉 2. Numéricas → reemplazar por 0 (como antes)
df[cols_numericas] = df[cols_numericas].fillna(0)

# Validación
nulos_despues = df.isnull().sum().sum()

print(f"🧹 Nulos después: {nulos_despues}")

assert nulos_despues == 0, "⚠️ Aún hay nulos en el dataset"

print("✅ Validación OK")

# Reporte QA
print(f"📊 Total imputados: {nulos_antes}")

🔎 Nulos antes: 1
🧹 Nulos después: 0
✅ Validación OK
📊 Total imputados: 1


/tmp/ipykernel_7638/3677371445.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(moda[0], inplace=True)


In [31]:
# @title 🔤 SECCIÓN 5 — Normalización de texto

cols_texto = df.select_dtypes(include=['object']).columns

def formatear_texto(x):
    if isinstance(x, str):
        return x.title()
    return x

for col in cols_texto:
    df[col] = df[col].apply(formatear_texto)

print("Columnas normalizadas:")
print(list(cols_texto))

Columnas normalizadas:
['provincia', 'departamento', 'sector', 'ambito']


In [32]:
# @title 📊 SECCIÓN 6 — Formateo de tasas (3 decimales)

cols_tasa = [col for col in df.columns if col.startswith("tasa_")]

df[cols_tasa] = df[cols_tasa].round(3)

print("Columnas de tasa formateadas:")
print(cols_tasa[:10])

Columnas de tasa formateadas:
['tasa_repitencia_grado_1', 'tasa_sobreedad_grado_1', 'tasa_repitencia_grado_2', 'tasa_sobreedad_grado_2', 'tasa_repitencia_grado_3', 'tasa_sobreedad_grado_3', 'tasa_repitencia_grado_4', 'tasa_sobreedad_grado_4', 'tasa_repitencia_grado_5', 'tasa_sobreedad_grado_5']


In [33]:
# @title 🚨 SECCIÓN 7 — Detección de Outliers

import numpy as np

numeric_df = df.select_dtypes(include=[np.number])

z_scores = np.abs((numeric_df - numeric_df.mean()) / numeric_df.std())

outliers = (z_scores > 3)

total_outliers = outliers.sum().sum()

print(f"🔎 Total outliers: {total_outliers}")

outliers_por_col = outliers.sum().sort_values(ascending=False)

print("\n📊 Top columnas con outliers:")
display(outliers_por_col.head(10))

# Ejemplos
cols = outliers_por_col[outliers_por_col > 0].index[:3]

for col in cols:
    print(f"\n🔍 Ejemplos en {col}")
    display(df.loc[outliers[col], [col]].head())

🔎 Total outliers: 26911

📊 Top columnas con outliers:


,0
tasa_sobreedad_grado_1,414
tasa_repitencia_grado_1,396
mujeres_anio_5,385
tasa_sobreedad_grado_2,385
total_anio_sup,382
mujeres_anio_6,375
varones_anio_1,373
mujeres_anio_4,369
total_anio_5,365
varones_anio_sup,362



🔍 Ejemplos en tasa_sobreedad_grado_1


,tasa_sobreedad_grado_1
89,40.000
349,42.623
353,31.066
357,30.000
363,33.456



🔍 Ejemplos en tasa_repitencia_grado_1


,tasa_repitencia_grado_1
21,18.182
73,14.286
111,14.328
115,20.000
121,21.739



🔍 Ejemplos en mujeres_anio_5


,mujeres_anio_5
13,2024
106,1751
133,1872
159,5706
160,3464


In [34]:
# @title 💾 SECCIÓN 8 — Exportación y descarga

from google.colab import files

file_name = "matricula_final_QA.xlsx"

df.to_excel(file_name, index=False)

files.download(file_name)

print("✅ Dataset exportado correctamente")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Dataset exportado correctamente


Tratamiento de Dataset Trayectoria por Sexo

In [18]:
# @title 🧪 SECCIÓN 1 — Introducción

from IPython.display import display, HTML

display(HTML("""
<h2 style='color:#2E86C1;'>Aduana de Calidad de Datos - Dataset Trayectoria</h2>
<p>Este proceso valida la calidad y consistencia del dataset de trayectoria educativa.</p>
<ul>
<li>Control de nulos</li>
<li>Normalización de texto</li>
<li>Detección de valores atípicos</li>
<li>Validación final para análisis estadístico</li>
</ul>
"""))

In [19]:
# @title 📥 SECCIÓN 2 — Carga del dataset

import pandas as pd
import numpy as np

from google.colab import files

print("📂 Subí el archivo trayectoria_limpia.xlsx")

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_excel(file_name)

print("✅ Dataset cargado correctamente")
df.head()

📂 Subí el archivo trayectoria_limpia.xlsx


Saving Trayectoria_Limpio.xlsx to Trayectoria_Limpio.xlsx
✅ Dataset cargado correctamente


,anio,provincia,departamento,sector,ambito,inscritos_marzo_grado_1,inscritos_marzo_grado_2,inscritos_marzo_grado_3,inscritos_marzo_grado_4,inscritos_marzo_grado_5,...,reprobados_mujeres_anio_2,reprobados_mujeres_anio_3,reprobados_mujeres_anio_4,reprobados_mujeres_anio_5,reprobados_mujeres_anio_6,reprobados_mujeres_anio_superior,egresados_primaria,egresados_mujeres_primaria,egresados_secundaria,egresados_mujeres_secundaria
0,2011,Buenos Aires,25 DE MAYO,Estatal,Rural,123,165,121,136,111,...,9,3,2,0,3,0,0,0,67,26
1,2011,Buenos Aires,25 DE MAYO,Estatal,Urbano,349,336,340,361,348,...,64,28,47,28,5,0,0,0,102,74
2,2011,Buenos Aires,25 DE MAYO,Privado,Urbano,152,169,149,132,132,...,5,0,3,0,6,0,0,0,27,19
3,2011,Buenos Aires,9 DE JULIO,Estatal,Rural,177,111,128,141,119,...,13,3,1,0,5,0,0,0,27,13
4,2011,Buenos Aires,9 DE JULIO,Estatal,Urbano,568,543,546,578,512,...,82,57,58,8,31,0,0,0,209,116


In [20]:
# @title 🔍 SECCIÓN 3 — Análisis de nulos

from IPython.display import display, HTML

# Convertir vacíos a NaN
df = df.replace(r'^\s*$', np.nan, regex=True)

nulls = df.isnull().sum().sort_values(ascending=False)
nulls = nulls[nulls > 0]

html = f"""
<div style='height:300px; overflow-y:scroll; border:1px solid #ccc; padding:10px;'>
<h3 style='color:#C0392B;'>Columnas con valores nulos</h3>
<ul>
{''.join([f"<li><b>{col}</b>: {val}</li>" for col, val in nulls.items()])}
</ul>
</div>
"""

display(HTML(html))

print(f"Total columnas con nulos: {len(nulls)}")
print(f"Total valores nulos: {df.isnull().sum().sum()}")

Total columnas con nulos: 1
Total valores nulos: 1


In [22]:
# @title 🧹 SECCIÓN 4 — Tratamiento de nulos (Imputación inteligente)

# Contar nulos antes
nulos_antes = df.isnull().sum().sum()
print(f"🔎 Nulos antes: {nulos_antes}")

# Separar columnas
cols_texto = df.select_dtypes(include=['object']).columns
cols_numericas = df.select_dtypes(include=[np.number]).columns

# 👉 1. Texto → reemplazar por valor más frecuente (moda)
for col in cols_texto:
    if df[col].isnull().sum() > 0:
        moda = df[col].mode()
        if not moda.empty:
            df[col].fillna(moda[0], inplace=True)

# 👉 2. Numéricas → reemplazar por 0 (como antes)
df[cols_numericas] = df[cols_numericas].fillna(0)

# Validación
nulos_despues = df.isnull().sum().sum()

print(f"🧹 Nulos después: {nulos_despues}")

assert nulos_despues == 0, "⚠️ Aún hay nulos en el dataset"

print("✅ Validación OK")

# Reporte QA
print(f"📊 Total imputados: {nulos_antes}")

🔎 Nulos antes: 0
🧹 Nulos después: 0
✅ Validación OK
📊 Total imputados: 0


In [25]:
# @title 🔤 SECCIÓN 5 — Normalización de texto

cols_texto = df.select_dtypes(include=['object']).columns

def formatear_texto(x):
    if isinstance(x, str):
        return x.title()
    return x

for col in cols_texto:
    df[col] = df[col].apply(formatear_texto)

print("Columnas normalizadas:")
print(list(cols_texto))

Columnas normalizadas:
['provincia', 'departamento', 'sector', 'ambito']


In [26]:
# @title 🚨 SECCIÓN 6 — Detección de Outliers

import numpy as np

numeric_df = df.select_dtypes(include=[np.number])

z_scores = np.abs((numeric_df - numeric_df.mean()) / numeric_df.std())

outliers = (z_scores > 3)

total_outliers = outliers.sum().sum()

print(f"🔎 Total de valores atípicos detectados: {total_outliers}")

outliers_por_col = outliers.sum().sort_values(ascending=False)

print("\n📊 Columnas con más outliers:")
display(outliers_por_col.head(10))

# Mostrar ejemplos reales
cols = outliers_por_col[outliers_por_col > 0].index[:3]

for col in cols:
    print(f"\n🔍 Ejemplos de outliers en {col}:")
    display(df.loc[outliers[col], [col]].head())

🔎 Total de valores atípicos detectados: 51292

📊 Columnas con más outliers:


,0
inscritos_marzo_anio_superior,390
alumnos_diciembre_anio_superior,387
alumnos_diciembre_mujeres_anio_5,386
inscritos_marzo_mujeres_anio_5,383
aprobados_mujeres_anio_6,381
inscritos_marzo_mujeres_anio_6,380
inscritos_marzo_mujeres_anio_superior,378
alumnos_diciembre_mujeres_anio_6,376
egresados_mujeres_secundaria,375
alumnos_diciembre_mujeres_anio_superior,374



🔍 Ejemplos de outliers en inscritos_marzo_anio_superior:


,inscritos_marzo_anio_superior
396,409
435,412
441,380
447,414
465,362



🔍 Ejemplos de outliers en alumnos_diciembre_anio_superior:


,alumnos_diciembre_anio_superior
396,423
435,386
441,365
447,396
465,354



🔍 Ejemplos de outliers en alumnos_diciembre_mujeres_anio_5:


,alumnos_diciembre_mujeres_anio_5
133,1711
159,5526
160,2995
161,1707
180,2139


In [27]:
# @title 💾 SECCIÓN 7 — Exportación y descarga

from google.colab import files

file_name = "trayectoria_final_QA.xlsx"

df.to_excel(file_name, index=False)

files.download(file_name)

print("✅ Dataset exportado correctamente")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Dataset exportado correctamente
